In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
data = pd.read_csv('insurance.csv')

In [5]:
data.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## Datanın hazırlanması - Data Preprocessing

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [7]:
data.isna().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [8]:
target = 'charges'

In [9]:
X = data.drop(columns=[target])
y = data[target]

In [10]:
# Datanın bölünməsi - Splitting the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

In [11]:
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

In [12]:
#For Tree/Forest/XGB: numeric passthrough (no scaling), one-hot for categoricals
preprocess_basic = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop",
)
preprocess_basic

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,True


In [16]:
# For GLM/KNN/SVM: scaling for numeric, one-hot for categoricals
preprocess_scaled = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("scaler", StandardScaler())]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop",
)
preprocess_scaled

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


## Alqoritmalar

In [13]:
def eval_reg(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    print(f"\n[{name}]")
    print(f"MAE : {mae:,.4f}")
    print(f"RMSE: {rmse:,.4f}")
    print(f"R^2 : {r2:,.4f}")
    return {"model": name, "mae": mae, "rmse": rmse, "rsq": r2}

In [14]:
results = []

### GLM — Linear Regression

In [17]:
glm_pipe = Pipeline([
    ("prep", preprocess_scaled),
    ("model", LinearRegression())
])
results.append(eval_reg("GLM (Linear Regression)", glm_pipe, X_train, y_train, X_test, y_test))


[GLM (Linear Regression)]
MAE : 4,013.1006
RMSE: 30,552,437.4165
R^2 : 0.8002


### KNN

In [23]:
knn_pipe = Pipeline([
    ("prep", preprocess_scaled),
    ("model", KNeighborsRegressor(n_neighbors=3))
])
results.append(eval_reg("KNN", knn_pipe, X_train, y_train, X_test, y_test))


[KNN]
MAE : 3,350.9792
RMSE: 30,775,374.8708
R^2 : 0.7987


### SVM

In [24]:
svm_pipe = Pipeline([
    ("prep", preprocess_scaled),
    ("model", SVR(kernel="rbf"))
])
results.append(eval_reg("SVM", svm_pipe, X_train, y_train, X_test, y_test))


[SVM]
MAE : 8,379.2165
RMSE: 170,860,340.5032
R^2 : -0.1175


### Decision Tree

In [25]:
tree_pipe = Pipeline([
    ("prep", preprocess_basic),
    ("model", DecisionTreeRegressor(
        max_depth=15, #the maximum number of splits from root to leaf
                      #large value: high variance, can overfit
                      #small value: high bias, low variance
        min_samples_split=2, #minimum row count in a node
                             #large value: less overfitting, high bias, low variance
                             #smal value: can overfit, low bias, high variance
        random_state=123))
])
results.append(eval_reg("Decision Tree", tree_pipe, X_train, y_train, X_test, y_test))


[Decision Tree]
MAE : 3,027.1570
RMSE: 41,856,750.2637
R^2 : 0.7262


### Random Forest (Bagging)

In [ ]:
rf_pipe = Pipeline([
    ("prep", preprocess_basic),
    ("model", RandomForestRegressor(
        n_estimators=2000,
        min_samples_leaf=5,
        random_state=123
    ))
])
results.append(eval_reg("Random Forest", rf_pipe, X_train, y_train, X_test, y_test))


[Random Forest]
MAE : 2,319.9497
RMSE: 15,427,144.2546
R^2 : 0.8991


### XGBoost (Boosting)

In [51]:
xgb_pipe = Pipeline([
    ("prep", preprocess_basic),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        n_estimators=100, #more trees: higher capacity and training time. Usually paired with a smaller learn_rate
        learning_rate=0.05, #step size reduction per boosting step
                            #small (0.01–0.1): slower, steadier learning, needs more trees, often better generalization
                            #large (0.2–0.3): faster, risk of overfit if trees is also large
        max_depth=3, #the maximum number of splits from root to leaf
                     #large (6–12): high variance, can overfit
                     #small (3–6): high bias, low variance
        min_child_weight=15, #large (5–20): less overfitting, high bias, low variance
                            #smal (1–2): can overfit, low bias, high variance
        random_state=42
    ))
])
results.append(eval_reg("XGBoost", xgb_pipe, X_train, y_train, X_test, y_test))


[XGBoost]
MAE : 2,157.9512
RMSE: 13,511,460.6744
R^2 : 0.9116


### LightGBM

In [63]:
lgb_pipe = Pipeline([
    ("prep", preprocess_basic),
    ("model", LGBMRegressor(
        objective="regression",
        n_estimators=100,
        learning_rate=0.05,
        max_depth=8,
        min_child_samples=5,
        random_state=123
    ))
])

results.append(eval_reg("LightGBM", lgb_pipe, X_train, y_train, X_test, y_test))


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000115 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 324
[LightGBM] [Info] Number of data points in the train set: 1070, number of used features: 11
[LightGBM] [Info] Start training from score 13189.257679
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

[LightGBM]
MAE : 2,162.7693
RMSE: 14,878,029.3752
R^2 : 0.9027


c:\Users\tarlan.cabiyev\AppData\Local\anaconda3\envs\ayna\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


## Ən yaxşı model seçimi

In [65]:
pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)

,model,mae,rmse,rsq
0,XGBoost,2157.951199,1.351146e+07,0.911629
1,XGBoost,2157.951199,1.351146e+07,0.911629
2,XGBoost,2157.951199,1.351146e+07,0.911629
3,XGBoost,2146.319890,1.375511e+07,0.910036
4,XGBoost,2185.265618,1.425289e+07,0.906780
5,XGBoost,2387.867709,1.443665e+07,0.905578
6,XGBoost,2235.392509,1.479208e+07,0.903254
7,LightGBM,2162.769261,1.487803e+07,0.902692
8,LightGBM,2162.769261,1.487803e+07,0.902692
9,LightGBM,2162.769261,1.487803e+07,0.902692
